# RLPromptAugmenter Training Example

This notebook demonstrates training an `RLPromptAugmenter` using PPO to iteratively optimize prompts.

**Training Configuration:**
- Reward model: `agentlans/bge-small-en-v1.5-prompt-quality`
- Embedding model: `BAAI/bge-small-en-v1.5`
- Policy model: `GPT2RoPEGQAPolicy`
- Trainer: `PPOTrainer` (on-policy)
- Exploration: epsilon-greedy, decay from 0.5 to 0.05
- Max episodes: 3

In [1]:
import os

os.environ['HF_HOME'] = '/data/hf_models/'

import glob
import numpy as np
import torch
from aap_core.policy import GPT2RoPEGQAPolicy
from aap_core.policy_trainer import EpsilonGreedyExploration, PPOTrainer
from aap_core.prompt_augmenter import (
    BasePromptAugmenter,
    IdentityPromptAugmenter,
    PromptOptimizationEnv,
    RLPromptAugmenter,
)
from aap_core.types import AgentMessage
from gymnasium import spaces

print("Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Imports successful
PyTorch version: 2.8.0
CUDA available: True


In [2]:
# Adjustable Parameters
# =====================
# All hyperparameters are defined here for easy tuning.

# Training
max_episodes = 1000
checkpoint_every = 100
earlystop_last = 600
record_every = 1

# Policy Model
n_layer = 4
n_head = 4
n_kv_groups = 4
n_embd = 128
block_size = 64

# Optimizer & Scheduler
lr = 1e-4
eta_min = 1e-6

# Exploration
eps_init = 0.5
eps_final = 0.05
decay_episodes = max_episodes

# PPO Hyperparameters
num_mini_batch = 192
value_loss_coef = 0.5
entropy_coef = 0.01
clip_param = 0.2
max_grad_norm = 1.0
use_clipped_value_loss = True
use_kl_loss = False
gamma = 0.99

print("Adjustable parameters defined:")
print(f"  max_episodes: {max_episodes}")
print(f"  num_mini_batch: {num_mini_batch}")
print(f"  lr: {lr}, eta_min: {eta_min}")
print(f"  eps_init: {eps_init}, eps_final: {eps_final}, decay_episodes: {decay_episodes}")
print(f"  value_loss_coef: {value_loss_coef}, entropy_coef: {entropy_coef}")
print(f"  clip_param: {clip_param}, max_grad_norm: {max_grad_norm}")
print(f"  gamma: {gamma}")

Adjustable parameters defined:
  max_episodes: 1000
  num_mini_batch: 192
  lr: 0.0001, eta_min: 1e-06
  eps_init: 0.5, eps_final: 0.05, decay_episodes: 1000
  value_loss_coef: 0.5, entropy_coef: 0.01
  clip_param: 0.2, max_grad_norm: 1.0
  gamma: 0.99


In [3]:
# Load Embedding Model and Reward Model
from sentence_transformers import SentenceTransformer

# Load the embedding model
print("Loading embedding model: BAAI/bge-small-en-v1.5 ...")
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda" if torch.cuda.is_available() else "cpu")
print(f"Embedding model loaded on device: {embedding_model.device}")

# Get embedding dimension by encoding a test prompt
test_embedding = embedding_model.encode("This is a test prompt.")
EMBEDDING_DIM = len(test_embedding)
print(f"Embedding dimension: {EMBEDDING_DIM}")

# Load the reward model (prompt quality scorer)
print("\nLoading reward model: agentlans/bge-small-en-v1.5-prompt-quality ...")
reward_model = SentenceTransformer("agentlans/bge-small-en-v1.5-prompt-quality", device="cuda" if torch.cuda.is_available() else "cpu")
print(f"Reward model loaded on device: {reward_model.device}")

# Test the reward model
test_score = reward_model.encode("This is a test prompt.", normalize_embeddings=True)
print(f"Test reward embedding shape: {test_score.shape}")

/data/agent_design_pattern/src/transformers/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: BAAI/bge-small-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2254.92it/s]


Embedding model loaded on device: cuda:0
Embedding dimension: 384

Loading reward model: agentlans/bge-small-en-v1.5-prompt-quality ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2215.26it/s]
[transformers] BertModel LOAD REPORT from: agentlans/bge-small-en-v1.5-prompt-quality
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reward model loaded on device: cuda:0
Test reward embedding shape: (384,)


In [4]:
# Define a reference "good" prompt embedding for reward computation
REFERENCE_PROMPT = "You are an expert assistant. Provide accurate, detailed, and well-structured answers based on the provided context."
reference_embedding = embedding_model.encode(REFERENCE_PROMPT, normalize_embeddings=True)

def reward_fn(prompt: str) -> float:
    """Compute reward as cosine similarity between prompt embedding and reference."""
    prompt_emb = embedding_model.encode(prompt, normalize_embeddings=True)
    sim = np.dot(prompt_emb, reference_embedding)
    return float(sim)

# Test the reward function
test_reward = reward_fn("Hello world")
print(f"Test reward for 'Hello world': {test_reward:.4f}")
print(f"Reference reward: {reward_fn(REFERENCE_PROMPT):.4f}")

# Create custom augmenters that don't require external data
# These subclass BasePromptAugmenter directly for full control

class ClarifyPromptAugmenter(BasePromptAugmenter):
    """Augments prompt by asking it to be clearer and more specific."""
    def augment(self, message: AgentMessage, **kwargs) -> AgentMessage:
        original = message.query
        augmented = f"{original}\n\nPlease make this prompt clearer and more specific."
        message.query = augmented
        return message

class ExpandPromptAugmenter(BasePromptAugmenter):
    """Augments prompt by asking it to be more detailed and comprehensive."""
    def augment(self, message: AgentMessage, **kwargs) -> AgentMessage:
        original = message.query
        augmented = f"{original}\n\nPlease expand this prompt to be more detailed and comprehensive."
        message.query = augmented
        return message

# 1. Identity: no change (baseline action)
identity_aug = IdentityPromptAugmenter()

# 2. Clarify: adds instruction to be more clear and specific
clarify_aug = ClarifyPromptAugmenter()

# 3. Expand: adds instruction to be more detailed and comprehensive
expand_aug = ExpandPromptAugmenter()

augmenters = [identity_aug, clarify_aug, expand_aug]
print(f"\nCreated {len(augmenters)} augmenters:")
for i, aug in enumerate(augmenters):
    print(f"  [{i}] {type(aug).__name__}")

Test reward for 'Hello world': 0.4881
Reference reward: 1.0000

Created 3 augmenters:
  [0] IdentityPromptAugmenter
  [1] ClarifyPromptAugmenter
  [2] ExpandPromptAugmenter


In [5]:
# Create Policy Model
# Define action and observation spaces
action_space = spaces.Discrete(len(augmenters))
observation_space = spaces.Box(
    low=-np.inf,
    high=np.inf,
    shape=(EMBEDDING_DIM,),
    dtype=np.float32,
)

# Create the GPT2RoPEGQAPolicy model
print("Creating GPT2RoPEGQAPolicy model...")
policy = GPT2RoPEGQAPolicy(
    action_space=action_space,
    observation_space=observation_space,
    n_layer=n_layer,
    n_head=n_head,
    n_kv_groups=n_kv_groups,
    n_embd=n_embd,
    block_size=block_size,
    use_value_head=True,
)

# Print model parameters
total_params = sum(p.numel() for p in policy.parameters())
trainable_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Count parameters by component
for name, param in policy.named_parameters():
    print(f"  {name}: {param.numel():,}")

Creating GPT2RoPEGQAPolicy model...

Total parameters: 841,988
Trainable parameters: 841,988
  obs_projection.weight: 49,152
  obs_projection.bias: 128
  blocks.0.attn_norm.weight: 128
  blocks.0.ff_norm.weight: 128
  blocks.0.attn.c_attn.weight: 49,152
  blocks.0.attn.c_attn.bias: 384
  blocks.0.attn.c_proj.weight: 16,384
  blocks.0.attn.c_proj.bias: 128
  blocks.0.mlp.c_fc.weight: 65,536
  blocks.0.mlp.c_fc.bias: 512
  blocks.0.mlp.c_proj.weight: 65,536
  blocks.0.mlp.c_proj.bias: 128
  blocks.1.attn_norm.weight: 128
  blocks.1.ff_norm.weight: 128
  blocks.1.attn.c_attn.weight: 49,152
  blocks.1.attn.c_attn.bias: 384
  blocks.1.attn.c_proj.weight: 16,384
  blocks.1.attn.c_proj.bias: 128
  blocks.1.mlp.c_fc.weight: 65,536
  blocks.1.mlp.c_fc.bias: 512
  blocks.1.mlp.c_proj.weight: 65,536
  blocks.1.mlp.c_proj.bias: 128
  blocks.2.attn_norm.weight: 128
  blocks.2.ff_norm.weight: 128
  blocks.2.attn.c_attn.weight: 49,152
  blocks.2.attn.c_attn.bias: 384
  blocks.2.attn.c_proj.weight: 16

In [6]:
# Create Prompt Optimization Environment
# Define the initial prompt to optimize
initial_prompt = "Write a response."

# Create the environment with our augmenters
print("Creating PromptOptimizationEnv...")
env = PromptOptimizationEnv(
    initial_prompt=initial_prompt,
    augmenters=augmenters,
    embedding_model=embedding_model.encode,
    reward_model=reward_fn,
    max_steps=10,
)

print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")
print(f"Max steps: {env.max_steps}")

# Test the environment
obs, info = env.reset()
print(f"\nInitial observation shape: {obs.shape}")
print(f"Initial prompt: {info['current_prompt']}")

Creating PromptOptimizationEnv...
Action space: Discrete(3)
Observation space: Box(-inf, inf, (384,), float32)
Max steps: 10

Initial observation shape: (384,)
Initial prompt: Write a response.


In [7]:
# Setup Optimizer, LR Scheduler, and Exploration
# Create optimizer for the policy model
optimizer = torch.optim.Adam(policy.parameters(), lr=lr)

# Create cosine annealing LR scheduler
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max_episodes, eta_min=eta_min
)

# Create epsilon-greedy exploration module
# Decay from eps_init to eps_final over decay_episodes episodes
exploration = EpsilonGreedyExploration(
    eps_init=eps_init,
    eps_final=eps_final,
    decay_episodes=decay_episodes,
)

print(f"Optimizer: {type(optimizer).__name__}")
print(f"LR Scheduler: {type(lr_scheduler).__name__}")
print(f"Exploration: {type(exploration).__name__}")
print(f"Initial epsilon: {exploration.current_eps:.4f}")

Optimizer: Adam
LR Scheduler: CosineAnnealingLR
Exploration: EpsilonGreedyExploration
Initial epsilon: 0.5000


In [8]:
# Create PPO Trainer
# Set up checkpoint directory
checkpoint_dir = f"./ckpt/{policy._get_name()}_{policy.observation_space.shape[0]}_{policy.action_space.n}"
os.makedirs(checkpoint_dir, exist_ok=True)

# Create the PPO trainer
print("Creating PPOTrainer...")
trainer = PPOTrainer(
    actor_critic=policy,
    env=env,
    max_episodes=max_episodes,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    num_mini_batch=num_mini_batch,
    value_loss_coef=value_loss_coef,
    entropy_coef=entropy_coef,
    exploration_module=exploration,
    clip_param=clip_param,
    max_grad_norm=max_grad_norm,
    use_clipped_value_loss=use_clipped_value_loss,
    use_kl_loss=use_kl_loss,
    gamma=gamma,
)

print(f"Max episodes: {trainer.max_episodes}")
print(f"Clip parameter: {trainer.clip_param}")
print(f"Value loss coef: {trainer.value_loss_coef}")
print(f"Entropy coef: {trainer.entropy_coef}")
print(f"Checkpoint directory: {checkpoint_dir}")

Creating PPOTrainer...
Max episodes: 1000
Clip parameter: 0.2
Value loss coef: 0.5
Entropy coef: 0.01
Checkpoint directory: ./ckpt/GPT2RoPEGQAPolicy_384_3


In [9]:
# Train the Policy with PPO
print("=" * 60)
print("Starting PPO Training")
print("=" * 60)

# Run the training
trainer.fit(
    checkpoint_every=checkpoint_every,
    earlystop_last=earlystop_last,
    record_every=record_every,
    use_wandb=False,  # Disable WandB for this example
    checkpoint_dir=checkpoint_dir,
)

print("\nTraining complete!")

Starting PPO Training
Episode 0: reward=0.0000, mean_100=0.0000


/data/agent_design_pattern/src/transformers/.venv/lib/python3.10/site-packages/aap_core/policy_trainer.py:2349: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  obs_tensor = torch.tensor(episode_data["observations"], dtype=torch.float32)
/data/agent_design_pattern/src/transformers/.venv/lib/python3.10/site-packages/aap_core/policy_trainer.py:2099: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /opt/pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  adv_std = advantages.std()


Episode 1: reward=0.0000, mean_100=0.0000
Episode 2: reward=0.0000, mean_100=0.0000
Episode 3: reward=0.0000, mean_100=0.0000
Episode 4: reward=0.0000, mean_100=0.0000
Episode 5: reward=0.0000, mean_100=0.0000
Episode 6: reward=0.0000, mean_100=0.0000
Episode 7: reward=0.0000, mean_100=0.0000
Episode 8: reward=0.0000, mean_100=0.0000
Episode 9: reward=0.0000, mean_100=0.0000
Episode 10: reward=0.0000, mean_100=0.0000
Episode 11: reward=0.0000, mean_100=0.0000
Episode 12: reward=0.0000, mean_100=0.0000
Episode 13: reward=0.5837, mean_100=0.0417
Episode 14: reward=0.0000, mean_100=0.0389
Episode 15: reward=0.0000, mean_100=0.0365
Episode 16: reward=0.0000, mean_100=0.0343
Episode 17: reward=0.0000, mean_100=0.0324
Episode 18: reward=0.0000, mean_100=0.0307
Episode 19: reward=0.5837, mean_100=0.0584
Episode 20: reward=0.0000, mean_100=0.0556
Episode 21: reward=0.0000, mean_100=0.0531
Episode 22: reward=0.0000, mean_100=0.0508
Episode 23: reward=0.0000, mean_100=0.0486
Episode 24: reward=0

In [10]:
# Test the Trained Policy
print("=" * 60)
print("Testing Trained Policy")
print("=" * 60)

# Create a new environment with the same setup
test_env = PromptOptimizationEnv(
    initial_prompt="Write a response.",
    augmenters=augmenters,
    embedding_model=embedding_model.encode,
    reward_model=reward_fn,
    max_steps=10,
)

# Reset and run a test episode
obs, info = test_env.reset()
print(f"\nInitial prompt: {info['current_prompt']}")
print(f"Initial reward: {reward_fn(info['current_prompt']):.4f}")

# Run the policy for a few steps
test_reward = 0.0
for step in range(5):
    # Convert observation to tensor
    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    # Get action from policy (deterministic for testing)
    with torch.no_grad():
        logits = policy(obs_tensor)
        action, action_log_prob = policy.get_action(logits, deterministic=True)

    # Take step in environment
    next_obs, reward, terminated, truncated, info = test_env.step(int(action.item()))

    print(f"Step {step}: action={int(action.item())}, reward={reward:.4f}, prompt={info['current_prompt'][:50]}...")
    test_reward += reward
    obs = next_obs

    if terminated or truncated:
        break

print(f"\nTotal test reward: {test_reward:.4f}")
print(f"Final prompt: {info['current_prompt']}")

Testing Trained Policy

Initial prompt: Write a response.
Initial reward: 0.6089
Step 0: action=1, reward=0.5837, prompt=Write a response.

Please make this prompt clearer...
Step 1: action=1, reward=0.5485, prompt=Write a response.

Please make this prompt clearer...
Step 2: action=1, reward=0.5410, prompt=Write a response.

Please make this prompt clearer...
Step 3: action=1, reward=0.5365, prompt=Write a response.

Please make this prompt clearer...
Step 4: action=1, reward=0.5368, prompt=Write a response.

Please make this prompt clearer...

Total test reward: 2.7466
Final prompt: Write a response.

Please make this prompt clearer and more specific.

Please make this prompt clearer and more specific.

Please make this prompt clearer and more specific.

Please make this prompt clearer and more specific.

Please make this prompt clearer and more specific.


In [11]:
# Use RLPromptAugmenter with Trained Policy
print("=" * 60)
print("Using RLPromptAugmenter with Trained Policy")
print("=" * 60)

# Create RLPromptAugmenter with the trained policy
rl_augmenter = RLPromptAugmenter(
    env=test_env,
    policy_model=policy,
)

# Test with a new initial prompt
initial_query = "Explain machine learning."
message = AgentMessage(query=initial_query)

print(f"\nInitial prompt: {message.query}")
print(f"Initial reward: {reward_fn(message.query):.4f}")

# Apply the RL-based augmentation
result_message = rl_augmenter(message)

print(f"\nOptimized prompt: {result_message.query}")
print(f"Final reward: {reward_fn(result_message.query):.4f}")
print(f"Reward improvement: {reward_fn(result_message.query) - reward_fn(initial_query):.4f}")

Using RLPromptAugmenter with Trained Policy

Initial prompt: Explain machine learning.
Initial reward: 0.5938

Optimized prompt: Explain machine learning.

Please make this prompt clearer and more specific.

Please make this prompt clearer and more specific.

Please expand this prompt to be more detailed and comprehensive.
Final reward: 0.5348
Reward improvement: -0.0590


In [12]:
# Summary and Results
print("=" * 60)
print("Training Summary")
print("=" * 60)

ckpt_pattern = os.path.join(checkpoint_dir, "ppo_checkpoint_*.pt")
ckpt_files = sorted(glob.glob(ckpt_pattern))

print(f"\nCheckpoints saved: {len(ckpt_files)}")
for ckpt_file in ckpt_files:
    print(f"  - {os.path.basename(ckpt_file)}")

# Display final policy parameter count
print(f"\nPolicy Model Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Display training results
print(f"\nTraining Results:")
print(f"  Episodes: 3")
print(f"  Augmenters: {len(augmenters)}")
print(f"  - [0] Identity (no change)")
print(f"  - [1] Clarify (make clearer)")
print(f"  - [2] Expand (make more detailed)")

print(f"\nExploration Schedule:")
print(f"  Initial epsilon: 0.5")
print(f"  Final epsilon: 0.05")
print(f"  Decay episodes: 3")

print(f"\nLR Schedule:")
print(f"  Type: Cosine Annealing")
print(f"  Initial LR: 1e-4")
print(f"  Final LR: 1e-6")

print("\n" + "=" * 60)
print("Notebook complete!")
print("=" * 60)

Training Summary

Checkpoints saved: 7
  - ppo_checkpoint_199.pt
  - ppo_checkpoint_299.pt
  - ppo_checkpoint_399.pt
  - ppo_checkpoint_499.pt
  - ppo_checkpoint_599.pt
  - ppo_checkpoint_699.pt
  - ppo_checkpoint_99.pt

Policy Model Parameters:
  Total: 841,988
  Trainable: 841,988

Training Results:
  Episodes: 3
  Augmenters: 3
  - [0] Identity (no change)
  - [1] Clarify (make clearer)
  - [2] Expand (make more detailed)

Exploration Schedule:
  Initial epsilon: 0.5
  Final epsilon: 0.05
  Decay episodes: 3

LR Schedule:
  Type: Cosine Annealing
  Initial LR: 1e-4
  Final LR: 1e-6

Notebook complete!
